### References

*   [https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876](https://www.kaggle.com/code/abdmental01/jigsaw-mpnet-base-v2-inference-cv-0-876)
*   [https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo](https://www.kaggle.com/code/aerdem4/jigsaw-acrc-qwen7b-finetune-logits-processor-zoo)
*   [https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/](https://www.guruguru.science/competitions/24/discussions/21027ff1-2074-4e21-a249-b2d4170bd516/)
*   https://www.kaggle.com/code/mks2192/jigsaw-llama3-1-8b-instruct-training-one-epoch
*   [https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference](https://www.kaggle.com/code/fuumin621/qwen2-5-lora-finetune-baseline-inference)
*   https://www.kaggle.com/code/neibyr/30-min-just-use-semantic-search-qwen3-emb-0-6b
*   https://www.kaggle.com/code/datafan07/jigsaw-speed-run-10-min-triplet-and-faiss
*   https://www.kaggle.com/code/nahidhossainredom/deberta-v3-base-3-epochs-lb-0-906

In [ ]:
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'trl==0.21.0' 'optimum==1.27.0' 'auto-gptq==0.7.1' 'bitsandbytes==0.46.1' 'deepspeed==0.17.4' 'logits-processor-zoo==0.2.1' 'vllm==0.10.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'triton==3.2.0'
!uv pip install --system --no-index --find-links='/kaggle/input/jigsaw-packages2/whls/' 'clean-text'
!uv pip install --system --no-index -U --no-deps --find-links='/kaggle/input/jigsaw-packages2/whls/' 'peft' 'accelerate' 'datasets'

In [ ]:
%%writefile constants.py
BASE_MODEL_PATH = "/kaggle/input/qwen2.5/transformers/0.5b-instruct-gptq-int4/1"
LORA_PATH = "output/"
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules/"

POSITIVE_ANSWER = "Yes"
NEGATIVE_ANSWER = "No"
COMPLETE_PHRASE = "Answer:"
BASE_PROMPT = '''You are given a comment from reddit and a rule. Your task is to classify whether the comment violates the rule. Only respond Yes/No.'''

In [ ]:
%%writefile utils.py
import pandas as pd
from datasets import Dataset
from constants import POSITIVE_ANSWER, NEGATIVE_ANSWER, COMPLETE_PHRASE, BASE_PROMPT
import random, numpy as np
random.seed(42)
np.random.seed(42)


def build_prompt(row):
    return f"""
{BASE_PROMPT}

Subreddit: r/{row["subreddit"]}
Rule: {row["rule"]}
Examples:
1) {row["positive_example"]}
{COMPLETE_PHRASE} Yes

2) {row["negative_example"]}
{COMPLETE_PHRASE} No

---
Comment: {row["body"]}
{COMPLETE_PHRASE}"""


def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.5, random_state=42).reset_index(drop=True)

    flatten = []

    # ---------- 处理训练集 ----------
    train_df = train_dataset[["body", "rule", "subreddit", "rule_violation",
                              "positive_example_1","positive_example_2",
                              "negative_example_1","negative_example_2"]].copy()

    # 随机选 positive_example 和 negative_example
    train_df["positive_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["positive_example_1"],
        train_df["positive_example_2"]
    )
    train_df["negative_example"] = np.where(
        np.random.rand(len(train_df)) < 0.5,
        train_df["negative_example_1"],
        train_df["negative_example_2"]
    )

    # 删除原来的候选列
    train_df.drop(columns=["positive_example_1","positive_example_2",
                           "negative_example_1","negative_example_2"], inplace=True)

    flatten.append(train_df)

    # ---------- 处理测试集 ----------
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[["rule","subreddit",
                                        "positive_example_1","positive_example_2",
                                        "negative_example_1","negative_example_2"]].copy()

            if violation_type == "positive":
                # body 用当前 positive_example
                body_col = f"positive_example_{i}"
                other_positive_col = f"positive_example_{3-i}"  # 另一个 positive
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["positive_example"] = sub_dataset[other_positive_col]
                # negative_example 随机选
                sub_dataset["negative_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["negative_example_1"],
                    sub_dataset["negative_example_2"]
                )
                sub_dataset["rule_violation"] = 1

            else:  # violation_type == "negative"
                body_col = f"negative_example_{i}"
                other_negative_col = f"negative_example_{3-i}"
                sub_dataset["body"] = sub_dataset[body_col]
                sub_dataset["negative_example"] = sub_dataset[other_negative_col]
                sub_dataset["positive_example"] = np.where(
                    np.random.rand(len(sub_dataset)) < 0.5,
                    sub_dataset["positive_example_1"],
                    sub_dataset["positive_example_2"]
                )
                sub_dataset["rule_violation"] = 0

            # 删除原来的候选列
            sub_dataset.drop(columns=["positive_example_1","positive_example_2",
                                      "negative_example_1","negative_example_2"], inplace=True)

            flatten.append(sub_dataset)

    # 合并所有 DataFrame
    dataframe = pd.concat(flatten, axis=0)
    dataframe = dataframe.drop_duplicates(ignore_index=True)

    return dataframe



def build_dataset(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    columns = ["prompt"]
    if "rule_violation" in dataframe:
        dataframe["completion"] = dataframe["rule_violation"].map(
            {
                1: POSITIVE_ANSWER,
                0: NEGATIVE_ANSWER,
            }
        )
        columns.append("completion")

    dataframe = dataframe[columns]
    dataset = Dataset.from_pandas(dataframe)
    dataset.to_pandas().to_csv("/kaggle/working/dataset.csv", index=False)
    return dataset

In [ ]:
%%writefile train.py
import pandas as pd

from trl import SFTTrainer, SFTConfig
from peft import LoraConfig
from tqdm.auto import tqdm
from transformers.utils import is_torch_bf16_gpu_available
from utils import build_dataset, get_dataframe_to_train
from constants import DATA_PATH, BASE_MODEL_PATH, LORA_PATH


def main():
    dataframe = get_dataframe_to_train(DATA_PATH)
    train_dataset = build_dataset(dataframe)
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    
    training_args = SFTConfig(
        num_train_epochs=1,
        
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        
        optim="paged_adamw_8bit",
        learning_rate=1e-4, #keep high, lora usually likes high. 
        weight_decay=0.01,
        max_grad_norm=1.0,
        
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        
        bf16=is_torch_bf16_gpu_available(),
        fp16=not is_torch_bf16_gpu_available(),
        dataloader_pin_memory=True,
        
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    
        save_strategy="no",
        report_to="none",
    
        completion_only_loss=True,
        packing=False,
        remove_unused_columns=False,
    )
    
    trainer = SFTTrainer(
        BASE_MODEL_PATH,
        args=training_args,
        train_dataset=train_dataset,
        peft_config=lora_config,
    )
    
    trainer.train()
    trainer.save_model(LORA_PATH)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile inference.py
import os
os.environ["VLLM_USE_V1"] = "0"

import vllm
import torch
import pandas as pd
from logits_processor_zoo.vllm import MultipleChoiceLogitsProcessor
from vllm.lora.request import LoRARequest
from utils import build_dataset
from constants import BASE_MODEL_PATH, LORA_PATH, DATA_PATH, POSITIVE_ANSWER, NEGATIVE_ANSWER
import random
import multiprocessing as mp


def run_inference_on_device(df_slice):
    """在当前进程可见的 GPU 上跑 vLLM 推理"""
    llm = vllm.LLM(
        BASE_MODEL_PATH,
        quantization="gptq",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.98,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=2836,
        disable_log_stats=True,
        enable_prefix_caching=True,
        enable_lora=True,
        max_lora_rank=64,
    )

    tokenizer = llm.get_tokenizer()
    mclp = MultipleChoiceLogitsProcessor(tokenizer, choices=[POSITIVE_ANSWER, NEGATIVE_ANSWER])

    test_dataset = build_dataset(df_slice)
    texts = test_dataset["prompt"]

    outputs = llm.generate(
        texts,
        vllm.SamplingParams(
            skip_special_tokens=True,
            max_tokens=1,
            logits_processors=[mclp],
            logprobs=2,
        ),
        use_tqdm=True,
        lora_request=LoRARequest("default", 1, LORA_PATH)
    )

    log_probs = [
        {lp.decoded_token: lp.logprob for lp in out.outputs[0].logprobs[0].values()}
        for out in outputs
    ]
    predictions = pd.DataFrame(log_probs)[[POSITIVE_ANSWER, NEGATIVE_ANSWER]]
    predictions["row_id"] = df_slice["row_id"].values
    return predictions


def worker(device_id, df_slice, return_dict):
    # 限制该进程只看到一张 GPU
    os.environ["CUDA_VISIBLE_DEVICES"] = str(device_id)
    print(f"[Worker {device_id}] Running on GPU {device_id}, data size={len(df_slice)}")

    preds = run_inference_on_device(df_slice)
    return_dict[device_id] = preds


def main():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")

    # 随机选择例子
    test_dataframe["positive_example"] = test_dataframe.apply(
        lambda row: random.choice([row["positive_example_1"], row["positive_example_2"]]),
        axis=1
    )
    test_dataframe["negative_example"] = test_dataframe.apply(
        lambda row: random.choice([row["negative_example_1"], row["negative_example_2"]]),
        axis=1
    )
    test_dataframe = test_dataframe.drop(
        columns=["positive_example_1", "positive_example_2", "negative_example_1", "negative_example_2"],
        errors="ignore"
    )

    # 切分数据
    mid = len(test_dataframe) // 2
    df0 = test_dataframe.iloc[:mid].reset_index(drop=True)
    df1 = test_dataframe.iloc[mid:].reset_index(drop=True)

    manager = mp.Manager()
    return_dict = manager.dict()

    # 两个进程并行
    p0 = mp.Process(target=worker, args=(0, df0, return_dict))
    p1 = mp.Process(target=worker, args=(1, df1, return_dict))
    p0.start()
    p1.start()
    p0.join()
    p1.join()

    # 合并结果
    predictions = pd.concat([return_dict[0], return_dict[1]], ignore_index=True)

    # 构建 submission
    submission = predictions[["row_id", POSITIVE_ANSWER]].rename(columns={POSITIVE_ANSWER: "rule_violation"})
    rq = submission['rule_violation'].rank(method='average') / (len(submission) + 1)
    submission['rule_violation'] = rq

    submission.to_csv("submission_qwen.csv", index=False)
    print("✅ Saved submission_qwen.csv")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile accelerate_config.yaml
compute_environment: LOCAL_MACHINE
debug: false
deepspeed_config:
  gradient_accumulation_steps: 4
  gradient_clipping: 1.0
  train_batch_size: 64
  train_micro_batch_size_per_gpu: 4
  
  zero_stage: 2
  offload_optimizer_device: none
  offload_param_device: none
  zero3_init_flag: false
  
  stage3_gather_16bit_weights_on_model_save: false
  stage3_max_live_parameters: 1e8
  stage3_max_reuse_distance: 1e8
  stage3_prefetch_bucket_size: 5e7
  stage3_param_persistence_threshold: 1e5
  
  zero_allow_untested_optimizer: true
  zero_force_ds_cpu_optimizer: false
  
  fp16:
    enabled: true
    loss_scale: 0
    initial_scale_power: 16
    loss_scale_window: 1000
    hysteresis: 2
    min_loss_scale: 1
  
distributed_type: DEEPSPEED
downcast_bf16: 'no'
dynamo_config:
  dynamo_backend: INDUCTOR
  dynamo_use_fullgraph: false
  dynamo_use_dynamic: false
enable_cpu_affinity: false
machine_rank: 0
main_training_function: main
mixed_precision: fp16
num_machines: 1
num_processes: 2
rdzv_backend: static
same_network: true
tpu_env: []
tpu_use_cluster: false
tpu_use_sudo: false
use_cpu: false

In [ ]:
!accelerate launch --config_file accelerate_config.yaml train.py

In [ ]:
!python inference.py

In [ ]:
import os
import pandas as pd

In [ ]:
%%writefile constants.py
EMBDEDDING_MODEL_PATH = "/kaggle/input/qwen-3-embedding/transformers/0.6b/1"
MODEL_OUTPUT_PATH = '/kaggle/input/qwen3-8b-embedding'
DATA_PATH = "/kaggle/input/jigsaw-agile-community-rules"

# https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/blob/main/config_sentence_transformers.json
EMBEDDING_MODEL_QUERY = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"

CLEAN_TEXT = True
TOP_K = 2000
BATCH_SIZE = 128

In [ ]:
%%writefile utils.py
import pandas as pd
import torch.distributed as dist

from datasets import Dataset
from cleantext import clean
from tqdm.auto import tqdm

from constants import CLEAN_TEXT


def build_prompt(row):
    return f"""r/{row["subreddit"]}\nComment: {row["body"]}"""


def cleaner(text):
    return clean(
        text,
        fix_unicode=True,
        to_ascii=True,
        lower=False,
        no_line_breaks=False,
        no_urls=True,
        no_emails=True,
        no_phone_numbers=True,
        no_numbers=False,
        no_digits=False,
        no_currency_symbols=False,
        no_punct=False,
        replace_with_url="<URL>",
        replace_with_email="<EMAIL>",
        replace_with_phone_number="<PHONE>",
        lang="en",
    )



def get_dataframe_to_train(data_path):
    train_dataset = pd.read_csv(f"{data_path}/train.csv")
    test_dataset = pd.read_csv(f"{data_path}/test.csv").sample(frac=0.6, random_state=42).reset_index(drop=True)

    flatten = []
    flatten.append(train_dataset[["body", "rule", "subreddit", "rule_violation"]])
    
    for violation_type in ["positive", "negative"]:
        for i in range(1, 3):
            sub_dataset = test_dataset[[f"{violation_type}_example_{i}", "rule", "subreddit"]].copy()
            sub_dataset = sub_dataset.rename(columns={f"{violation_type}_example_{i}": "body"})
            sub_dataset["rule_violation"] = 1 if violation_type == "positive" else 0
            flatten.append(sub_dataset)

    dataframe = pd.concat(flatten, axis=0)    
    dataframe = dataframe.drop_duplicates(ignore_index=True)
    return dataframe


def prepare_dataframe(dataframe):
    dataframe["prompt"] = dataframe.apply(build_prompt, axis=1)

    
    if CLEAN_TEXT:
        tqdm.pandas(desc="cleaner")
        dataframe["prompt"] = dataframe["prompt"].progress_apply(cleaner)

    if "rule_violation" in dataframe.columns:
        dataframe["rule_violation"] = dataframe["rule_violation"].map(
            {
                1: 1,
                0: -1,
            }
        )

    return dataframe

In [ ]:
%%writefile semantic.py
import pandas as pd
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import semantic_search, dot_score
from tqdm.auto import tqdm
from peft import PeftModel, PeftConfig


from utils import get_dataframe_to_train, prepare_dataframe
from constants import DATA_PATH, EMBDEDDING_MODEL_PATH, EMBEDDING_MODEL_QUERY, TOP_K, BATCH_SIZE, MODEL_OUTPUT_PATH



def get_scores(test_dataframe):
    corpus_dataframe = get_dataframe_to_train(DATA_PATH)
    corpus_dataframe = prepare_dataframe(corpus_dataframe)
    
    # Load base model
    model = AutoModelForCausalLM.from_pretrained(EMBDEDDING_MODEL_PATH)
    tokenizer = AutoTokenizer.from_pretrained(EMBDEDDING_MODEL_PATH)
    
    # Load adapter configuration and model
    adapter_config = PeftConfig.from_pretrained(MODEL_OUTPUT_PATH)
    lora_model = PeftModel.from_pretrained(model, MODEL_OUTPUT_PATH, config=adapter_config)
    merged_model = lora_model.merge_and_unload()
    tokenizer.save_pretrained("Qwen3Emb_Finetuned")
    merged_model.save_pretrained("Qwen3Emb_Finetuned")

    # 4. Tạo lại SentenceTransformer từ encoder đã merge
    embedding_model = SentenceTransformer(model_name_or_path="Qwen3Emb_Finetuned", device="cuda")

    print('Done loading model!')

    result = []
    for rule in tqdm(test_dataframe["rule"].unique(), desc=f"Generate scores for each rule"):
        test_dataframe_part = test_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe.query("rule == @rule").reset_index(drop=True)
        corpus_dataframe_part = corpus_dataframe_part.reset_index(names="row_id")
        
        query_embeddings = embedding_model.encode(
            sentences=test_dataframe_part["prompt"].tolist(),
            prompt=EMBEDDING_MODEL_QUERY,
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        document_embeddings = embedding_model.encode(
            sentences=corpus_dataframe_part["prompt"].tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_tensor=True,
            device="cuda",
            normalize_embeddings=True,
        )
        test_dataframe_part["semantic"] = semantic_search(
            query_embeddings,
            document_embeddings,
            top_k=TOP_K,
            score_function=dot_score,
        )
        def get_score(semantic):
            semantic = pd.DataFrame(semantic)
            semantic = semantic.merge(
                corpus_dataframe_part[["row_id", "rule_violation"]],
                how="left",
                left_on="corpus_id",
                right_on="row_id",
            )
            semantic["score"] = semantic["score"]*semantic["rule_violation"]
            return semantic["score"].sum()
            
        tqdm.pandas(desc=f"Add label for {rule=}")
        test_dataframe_part["rule_violation"] = test_dataframe_part["semantic"].progress_apply(get_score)
        result.append(test_dataframe_part[["row_id", "rule_violation"]].copy())
        
    submission = pd.concat(result, axis=0)
    return submission


def generate_submission():
    test_dataframe = pd.read_csv(f"{DATA_PATH}/test.csv")
    test_dataframe = prepare_dataframe(test_dataframe)
    
    submission = get_scores(test_dataframe)
    submission = test_dataframe[["row_id"]].merge(submission, on="row_id", how="left")
    submission.to_csv("submission_qwen3.csv", index=False)


if __name__ == "__main__":
    generate_submission()

In [ ]:
!python semantic.py

In [ ]:
%%writefile triplet.py
#!/usr/bin/env python3

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import pandas as pd
import numpy as np
import random
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    models
)
from sentence_transformers.losses import TripletLoss
from sklearn.metrics.pairwise import cosine_similarity  # (원본 유지)
import re
from urllib.parse import urlparse
import faiss  # (원본 유지)
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Advanced clustering
from sklearn.cluster import AgglomerativeClustering
from umap import UMAP

# -----------------------------
# Helpers (원본 동일)
# -----------------------------
def cleaner(text):
    """Replace URLs with format: <url>: (domain/important-path)"""
    if not text:
        return text
    url_pattern = r'https?://[^\s<>"{}|\\^`\[\]]+'
    def replace_url(match):
        url = match.group(0)
        try:
            parsed = urlparse(url)
            domain = parsed.netloc.lower()
            if domain.startswith('www.'):
                domain = domain[4:]
            path_parts = [part for part in parsed.path.split('/') if part]
            if path_parts:
                important_path = '/'.join(path_parts[:2])
                return f"<url>: ({domain}/{important_path})"
            else:
                return f"<url>: ({domain})"
        except:
            return "<url>: (unknown)"
    return re.sub(url_pattern, replace_url, str(text))


def load_test_data():
    """Load test data."""
    print("Loading test data...")
    test_df = pd.read_csv('/kaggle/input/jigsaw-agile-community-rules/test.csv')
    print(f"Loaded {len(test_df)} test examples")
    print(f"Unique rules: {test_df['rule'].nunique()}")
    return test_df


def collect_all_texts(test_df):
    """Collect all unique texts from test set."""
    print("\nCollecting all texts for embedding...")
    all_texts = set()
    for body in test_df['body']:
        if pd.notna(body):
            all_texts.add(cleaner(str(body)))
    example_cols = ['positive_example_1', 'positive_example_2',
                    'negative_example_1', 'negative_example_2']
    for col in example_cols:
        for example in test_df[col]:
            if pd.notna(example):
                all_texts.add(cleaner(str(example)))
    all_texts = list(all_texts)
    print(f"Collected {len(all_texts)} unique texts")
    return all_texts


def generate_embeddings(texts, model, batch_size=64):
    """Generate BGE embeddings for all texts."""
    print(f"Generating embeddings for {len(texts)} texts...")
    embeddings = model.encode(
        sentences=texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_tensor=False,
        normalize_embeddings=True
    )
    return embeddings


def create_test_triplet_dataset(test_df, augmentation_factor=2, random_seed=42, subsample_fraction=1.0):
    """Create triplet dataset from test data: anchor=rule, positive=positive_example, negative=negative_example."""
    random.seed(random_seed)
    np.random.seed(random_seed)
    anchors, positives, negatives = [], [], []
    print("Creating rule-aligned triplets from test data...")
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test rows"):
        rule = cleaner(str(row['rule']))
        pos_examples = []
        neg_examples = []
        for neg_col in ['negative_example_1', 'negative_example_2']:  # Compliant → triplet positive
            if pd.notna(row[neg_col]):
                pos_examples.append(cleaner(str(row[neg_col])))
        for pos_col in ['positive_example_1', 'positive_example_2']:  # Violating → triplet negative
            if pd.notna(row[pos_col]):
                neg_examples.append(cleaner(str(row[pos_col])))
        for pos_ex in pos_examples:
            for neg_ex in neg_examples:
                anchors.append(rule)
                positives.append(pos_ex)
                negatives.append(neg_ex)

    if augmentation_factor > 0:
        print(f"Adding {augmentation_factor}x augmentation...")
        rule_positives = {}
        rule_negatives = {}
        for rule in test_df['rule'].unique():
            rule_df = test_df[test_df['rule'] == rule]
            pos_pool, neg_pool = [], []
            for _, row in rule_df.iterrows():
                for neg_col in ['negative_example_1', 'negative_example_2']:
                    if pd.notna(row[neg_col]):
                        pos_pool.append(cleaner(str(row[neg_col])))
                for pos_col in ['positive_example_1', 'positive_example_2']:
                    if pd.notna(row[pos_col]):
                        neg_pool.append(cleaner(str(row[pos_col])))
            rule_positives[rule] = list(set(pos_pool))
            rule_negatives[rule] = list(set(neg_pool))

        for rule in test_df['rule'].unique():
            clean_rule = cleaner(str(rule))
            pos_pool = rule_positives[rule]
            neg_pool = rule_negatives[rule]
            n_samples = min(augmentation_factor * len(pos_pool), len(pos_pool) * len(neg_pool))
            for _ in range(n_samples):
                if pos_pool and neg_pool:
                    anchors.append(clean_rule)
                    positives.append(random.choice(pos_pool))
                    negatives.append(random.choice(neg_pool))

    combined = list(zip(anchors, positives, negatives))
    random.shuffle(combined)
    original_count = len(combined)
    if subsample_fraction < 1.0:
        n_samples = int(len(combined) * subsample_fraction)
        combined = combined[:n_samples]
        print(f"Subsampled {original_count} -> {len(combined)} triplets ({subsample_fraction*100:.1f}%)")
    anchors, positives, negatives = zip(*combined) if combined else ([], [], [])
    print(f"Created {len(anchors)} triplets from test data")
    dataset = Dataset.from_dict({'anchor': list(anchors), 'positive': list(positives), 'negative': list(negatives)})
    return dataset


def fine_tune_model(model, train_dataset, epochs=3, batch_size=32, learning_rate=2e-5, margin=0.25, output_dir="./models/test-finetuned-bge"):
    """Fine-tune the sentence transformer model using triplet loss on test data."""
    print(f"Fine-tuning model on {len(train_dataset)} triplets...")
    loss = TripletLoss(model=model, triplet_margin=margin)
    dataset_size = len(train_dataset)
    steps_per_epoch = max(1, dataset_size // batch_size)
    max_steps = steps_per_epoch * epochs
    args = SentenceTransformerTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        warmup_steps=0,
        learning_rate=learning_rate,
        logging_steps=max(1, max_steps // 4),
        save_strategy="epoch",
        save_total_limit=1,
        fp16=True,  # 원본 유지(=GPU 환경 가정)
        max_grad_norm=1.0,
        dataloader_drop_last=False,
        gradient_checkpointing=True,
        gradient_accumulation_steps=1,
        max_steps=max_steps,
        report_to="none"
    )
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_dataset, loss=loss)
    trainer.train()
    final_model_path = f"{output_dir}/final"
    print(f"Saving fine-tuned model to {final_model_path}...")
    model.save_pretrained(final_model_path)
    return model, final_model_path


def load_or_create_finetuned_model(test_df):
    """Load fine-tuned model if exists, otherwise create and fine-tune it."""
    fine_tuned_path = "./models/test-finetuned-bge/final"
    if os.path.exists(fine_tuned_path):
        print(f"Loading existing fine-tuned model from {fine_tuned_path}...")
        try:
            word_embedding_model = models.Transformer(fine_tuned_path, max_seq_length=128, do_lower_case=True)
            pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
            model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
            print("Loaded fine-tuned model with explicit pooling")
        except:
            model = SentenceTransformer(fine_tuned_path)
            print("Loaded fine-tuned model with default configuration")
        model.half()  # 원본과 동일하게 half()
        return model

    print("Fine-tuned model not found. Creating new one...")
    print("Loading base BGE embedding model...")
    try:
        model_path = "/kaggle/input/baai/transformers/bge-base-en-v1.5/1"
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from Kaggle path with explicit pooling")
    except:
        model_path = ""  # BAAI/bge-small-en-v1.5
        word_embedding_model = models.Transformer(model_path, max_seq_length=256, do_lower_case=True)
        pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode="mean")
        base_model = SentenceTransformer(modules=[word_embedding_model, pooling_model])
        print("Loaded base model from local path with explicit pooling")

    triplet_dataset = create_test_triplet_dataset(test_df, augmentation_factor=2, subsample_fraction=1.)
    fine_tuned_model, model_path = fine_tune_model(
        model=base_model,
        train_dataset=triplet_dataset,
        epochs=1,
        batch_size=16,
        learning_rate=2e-5,
        margin=0.25
    )
    print(f"Fine-tuning completed. Model saved to: {model_path}")
    fine_tuned_model.half()  # 원본과 동일
    return fine_tuned_model


def generate_rule_embeddings(test_df, model):
    """Generate embeddings for each unique rule."""
    print("Generating rule embeddings...")
    unique_rules = test_df['rule'].unique()
    rule_embeddings = {}
    for rule in unique_rules:
        clean_rule = cleaner(str(rule))
        rule_emb = model.encode(clean_rule, convert_to_tensor=False, normalize_embeddings=True)
        rule_embeddings[rule] = rule_emb
    print(f"Generated embeddings for {len(rule_embeddings)} rules")
    return rule_embeddings


# -----------------------------
# 핵심 수정: UMAP crash 회피(최소 변경)
#   - 원본 조건(샘플 수 > 10 그리고 > 32) 유지
#   - 실제 UMAP 호출 시, n_components는 min(32, N-2)로만 clamp
#   - spectral init(기본값) 유지 → 원본과 가장 가깝게
# -----------------------------
def create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings):
    """Create centroids using Hierarchical Clustering + UMAP for better cluster representation."""
    print(f"\nCreating rule centroids with Hierarchical Clustering + UMAP...")
    base_umap_components = 32
    rule_centroids = {}

    for rule in test_df['rule'].unique():
        rule_data = test_df[test_df['rule'] == rule]

        pos_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['positive_example_1', 'positive_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        pos_embeddings.append(text_to_embedding[clean_text])

        neg_embeddings = []
        for _, row in rule_data.iterrows():
            for col in ['negative_example_1', 'negative_example_2']:
                if pd.notna(row[col]):
                    clean_text = cleaner(str(row[col]))
                    if clean_text in text_to_embedding:
                        neg_embeddings.append(text_to_embedding[clean_text])

        if pos_embeddings and neg_embeddings:
            pos_embeddings = np.array(pos_embeddings)
            neg_embeddings = np.array(neg_embeddings)

            # --- 원본 조건 유지: n > 10 and n > 32 일 때만 UMAP ---
            # 단, 호출 직전에 n_components를 N-2로 clamp하여 k>=N 에러 회피
            def maybe_umap(X):
                n = X.shape[0]
                if n > 10 and n > base_umap_components:
                    n_components_safe = min(base_umap_components, max(2, n - 2))
                    reducer = UMAP(n_components=n_components_safe, random_state=42)  # init='spectral'(default)
                    return reducer.fit_transform(X)
                else:
                    return X

            pos_reduced = maybe_umap(pos_embeddings)
            neg_reduced = maybe_umap(neg_embeddings)

            # Agglomerative clustering (원본 동일)
            n_pos_clusters = min(3, len(pos_embeddings))
            n_neg_clusters = min(3, len(neg_embeddings))

            pos_centroids = []
            neg_centroids = []

            if n_pos_clusters > 1:
                pos_clusterer = AgglomerativeClustering(n_clusters=n_pos_clusters)
                pos_labels = pos_clusterer.fit_predict(pos_reduced)
                for cluster_id in np.unique(pos_labels):
                    cluster_mask = pos_labels == cluster_id
                    cluster_embeddings = pos_embeddings[cluster_mask]
                    cluster_centroid = cluster_embeddings.mean(axis=0)
                    cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                    pos_centroids.append(cluster_centroid)
            else:
                pos_centroid = pos_embeddings.mean(axis=0)
                pos_centroid = pos_centroid / np.linalg.norm(pos_centroid)
                pos_centroids.append(pos_centroid)

            if n_neg_clusters > 1:
                neg_clusterer = AgglomerativeClustering(n_clusters=n_neg_clusters)
                neg_labels = neg_clusterer.fit_predict(neg_reduced)
                for cluster_id in np.unique(neg_labels):
                    cluster_mask = neg_labels == cluster_id
                    cluster_embeddings = neg_embeddings[cluster_mask]
                    cluster_centroid = cluster_embeddings.mean(axis=0)
                    cluster_centroid = cluster_centroid / np.linalg.norm(cluster_centroid)
                    neg_centroids.append(cluster_centroid)
            else:
                neg_centroid = neg_embeddings.mean(axis=0)
                neg_centroid = neg_centroid / np.linalg.norm(neg_centroid)
                neg_centroids.append(neg_centroid)

            rule_centroids[rule] = {
                'positive_centroids': pos_centroids,
                'negative_centroids': neg_centroids,
                'pos_count': len(pos_embeddings),
                'neg_count': len(neg_embeddings),
                'rule_embedding': rule_embeddings[rule]
            }

            print(f"  Rule: {rule[:50]}... - Pos: {len(pos_embeddings)}, Neg: {len(neg_embeddings)} - Clusters: Pos={len(pos_centroids)}, Neg={len(neg_centroids)}")

    print(f"Created hierarchical centroids for {len(rule_centroids)} rules")
    return rule_centroids


def predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids):
    """Predict test set using hierarchical clustering centroids and distance metrics."""
    print("\nMaking predictions on test set with Hierarchical Clustering centroids...")
    row_ids = []
    predictions = []
    for rule in test_df['rule'].unique():
        print(f"  Processing rule: {rule[:50]}...")
        rule_data = test_df[test_df['rule'] == rule]
        if rule not in rule_centroids:
            continue
        pos_centroids = rule_centroids[rule]['positive_centroids']
        neg_centroids = rule_centroids[rule]['negative_centroids']
        for _, row in rule_data.iterrows():
            body = cleaner(str(row['body']))
            row_id = row['row_id']
            if body not in text_to_embedding:
                continue
            body_embedding = text_to_embedding[body]
            pos_distances = []
            for pos_centroid in pos_centroids:
                distance = np.linalg.norm(body_embedding - pos_centroid)
                pos_distances.append(distance)
            neg_distances = []
            for neg_centroid in neg_centroids:
                distance = np.linalg.norm(body_embedding - neg_centroid)
                neg_distances.append(distance)
            min_pos_distance = min(pos_distances) if pos_distances else 1.0
            min_neg_distance = min(neg_distances) if neg_distances else 1.0
            rule_prediction = min_neg_distance - min_pos_distance
            row_ids.append(row_id)
            predictions.append(rule_prediction)
    print(f"Made predictions for {len(predictions)} test examples")
    return row_ids, np.array(predictions)


def main():
    print("="*70)
    print("IMPROVED SIMILARITY CLASSIFIER - HIERARCHICAL CLUSTERING + UMAP (same results as your script)")
    print("="*70)

    test_df = load_test_data()

    print("\n" + "="*50)
    print("MODEL PREPARATION PHASE")
    print("="*50)
    model = load_or_create_finetuned_model(test_df)

    all_texts = collect_all_texts(test_df)

    print("\n" + "="*50)
    print("EMBEDDING GENERATION PHASE")
    print("="*50)
    all_embeddings = generate_embeddings(all_texts, model)

    text_to_embedding = {text: emb for text, emb in zip(all_texts, all_embeddings)}

    rule_embeddings = generate_rule_embeddings(test_df, model)

    # 동일 로직 + UMAP crash만 회피
    rule_centroids = create_rule_centroids_with_hierarchical_clustering(test_df, text_to_embedding, rule_embeddings)

    print("\n" + "="*50)
    print("PREDICTION PHASE")
    print("="*50)
    row_ids, predictions = predict_test_set_with_hierarchical_clustering(test_df, text_to_embedding, rule_centroids)

    submission_df = pd.DataFrame({'row_id': row_ids, 'rule_violation': predictions})
    submission_df.to_csv('Triplet_submission.csv', index=False)

    print(f"\nSaved predictions for {len(submission_df)} test examples to submission.csv and Triplet_submission.csv")

    print(f"\n{'='*70}")
    print(f"HIERARCHICAL CLUSTERING + UMAP INFERENCE COMPLETED")
    print(f"Model: Fine-tuned BGE on test data triplets")
    print(f"Method: Hierarchical clustering + UMAP dimensionality reduction")
    print(f"Predicted on {len(test_df)} test examples")
    print(f"Prediction stats: min={predictions.min():.4f}, max={predictions.max():.4f}, mean={predictions.mean():.4f}")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile deberta.py
import os
import re
import pandas as pd
import numpy as np
import random
import torch
from urllib.parse import urlparse
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

def url_to_semantics(text):
    if not isinstance(text, str):
        return ""
    urls = re.findall(r'https?://[^\s/$.?#].[^\s]*', text)
    if not urls:
        return ""
    all_semantics = []
    seen = set()
    for url in urls:
        u = url.lower()
        m = re.search(r"(?:https?://)?([a-z0-9\-\.]+)\.[a-z]{2,}", u)
        if m:
            parts = m.group(1).split(".")
            for p in parts:
                if p and len(p) > 3 and p not in seen:
                    all_semantics.append(f"domain:{p}")
                    seen.add(p)
        path = re.sub(r"^(?:https?://)?[a-z0-9\.-]+\.[a-z]{2,}/?", "", u)
        parts = [p for p in re.split(r'[/_.-]+', path) if p and p.isalnum()]
        for p in parts:
            pc = re.sub(r"\.(html?|php|asp|jsp)$|#.*|\?.*", "", p)
            if pc and len(pc) > 3 and pc not in seen:
                all_semantics.append(f"path:{pc}")
                seen.add(pc)
    if not all_semantics:
        return ""
    return "\nURL Keywords: " + " ".join(all_semantics)

def get_dataframe_to_train(data_path):
    train_df = pd.read_csv(f"{data_path}/train.csv")
    test_df = pd.read_csv(f"{data_path}/test.csv")
    out = []
    for k in ["positive","negative"]:
        for i in range(1,3):
            c = f"{k}_example_{i}"
            if c in train_df.columns:
                sub = train_df[[c,"rule","subreddit"]].copy()
                sub = sub.rename(columns={c:"body"})
                sub["rule_violation"] = 1 if k=="positive" else 0
                sub = sub.dropna(subset=["body"])
                sub = sub[sub["body"].str.strip().str.len()>0]
                if len(sub):
                    out.append(sub)
    for k in ["positive","negative"]:
        for i in range(1,3):
            c = f"{k}_example_{i}"
            if c in test_df.columns:
                sub = test_df[[c,"rule","subreddit"]].copy()
                sub = sub.rename(columns={c:"body"})
                sub["rule_violation"] = 1 if k=="positive" else 0
                sub = sub.dropna(subset=["body"])
                sub = sub[sub["body"].str.strip().str.len()>0]
                if len(sub):
                    out.append(sub)
    df = pd.concat(out, axis=0) if out else pd.DataFrame(columns=["body","rule","subreddit","rule_violation"])
    df = df.drop_duplicates(subset=["body","rule","subreddit"], ignore_index=True)
    df = df.drop_duplicates(subset=["body","rule"], keep="first")
    return df.sample(frac=1, random_state=42).reset_index(drop=True)

def seed_everything(s=42):
    random.seed(s)
    os.environ["PYTHONHASHSEED"]=str(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

class CFG:
    model_name_or_path="/kaggle/input/huggingfacedebertav3variants/deberta-v3-base"
    data_path="/kaggle/input/jigsaw-agile-community-rules/"
    output_dir="./deberta_v3_small_final_model"
    EPOCHS=3
    LEARNING_RATE=2e-5
    MAX_LENGTH=512
    BATCH_SIZE=8
    N_SPLITS=5
    RANDOM_STATE=42

class JigsawDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings=encodings
        self.labels=labels
    def __getitem__(self, idx):
        item={k:torch.tensor(v[idx]) for k,v in self.encodings.items()}
        if self.labels is not None:
            item["labels"]=torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.encodings["input_ids"])

def build_inputs(df):
    df=df.copy()
    df["body_with_url"]=df["body"].apply(lambda x: x+url_to_semantics(x))
    df["input_text"]=df["rule"]+"[SEP]"+df["body_with_url"]
    return df

def tokenize(tokenizer, texts, max_len):
    return tokenizer(texts, truncation=True, padding=True, max_length=max_len)

def train_and_predict_submission():
    seed_everything(42)
    df=get_dataframe_to_train(CFG.data_path)
    df=build_inputs(df)
    test_df=pd.read_csv(f"{CFG.data_path}/test.csv")
    test_df=build_inputs(test_df)
    tokenizer=AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    tr_enc=tokenize(tokenizer, df["input_text"].tolist(), CFG.MAX_LENGTH)
    tr_lbl=df["rule_violation"].tolist()
    tr_ds=JigsawDataset(tr_enc, tr_lbl)
    model=AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
    args=TrainingArguments(output_dir=CFG.output_dir, num_train_epochs=CFG.EPOCHS, learning_rate=CFG.LEARNING_RATE, per_device_train_batch_size=CFG.BATCH_SIZE, warmup_ratio=0.1, weight_decay=0.01, report_to="none", save_strategy="no", logging_steps=10)
    trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
    trainer.train()
    te_enc=tokenize(tokenizer, test_df["input_text"].tolist(), CFG.MAX_LENGTH)
    te_ds=JigsawDataset(te_enc, None)
    preds=trainer.predict(te_ds)
    probs=torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=1)[:,1].numpy()
    sub=pd.DataFrame({"row_id":test_df["row_id"],"rule_violation":probs})
    sub.to_csv("deberta_submission.csv", index=False)

def evaluate_fold(trainer, ds, y_true):
    out=trainer.predict(ds)
    probs=torch.nn.functional.softmax(torch.tensor(out.predictions), dim=1)[:,1].numpy()
    preds=out.predictions.argmax(axis=1).astype(int)
    p,r,f1,_=precision_recall_fscore_support(y_true, preds, average="binary")
    auc=roc_auc_score(y_true, probs)
    return dict(precision=p, recall=r, f1=f1, auc=auc)

def cross_validation():
    seed_everything(CFG.RANDOM_STATE)
    df=get_dataframe_to_train(CFG.data_path)
    df=build_inputs(df)
    skf=StratifiedKFold(n_splits=CFG.N_SPLITS, shuffle=True, random_state=CFG.RANDOM_STATE)
    tokenizer=AutoTokenizer.from_pretrained(CFG.model_name_or_path)
    scores=[]
    for i,(tr_idx,va_idx) in enumerate(skf.split(df, df["rule"]),1):
        tr=df.iloc[tr_idx].reset_index(drop=True)
        va=df.iloc[va_idx].reset_index(drop=True)
        tr_enc=tokenize(tokenizer, tr["input_text"].tolist(), CFG.MAX_LENGTH)
        va_enc=tokenize(tokenizer, va["input_text"].tolist(), CFG.MAX_LENGTH)
        tr_ds=JigsawDataset(tr_enc, tr["rule_violation"].tolist())
        va_ds=JigsawDataset(va_enc, va["rule_violation"].tolist())
        model=AutoModelForSequenceClassification.from_pretrained(CFG.model_name_or_path, num_labels=2)
        args=TrainingArguments(output_dir=f"{CFG.output_dir}_fold{i}", num_train_epochs=CFG.EPOCHS, learning_rate=CFG.LEARNING_RATE, per_device_train_batch_size=CFG.BATCH_SIZE, warmup_ratio=0.1, weight_decay=0.01, report_to="none", save_strategy="no", logging_steps=20)
        trainer=Trainer(model=model, args=args, train_dataset=tr_ds)
        trainer.train()
        res=evaluate_fold(trainer, va_ds, va["rule_violation"].values)
        scores.append(res)
        print(f"Fold {i} AUC={res['auc']:.4f} F1={res['f1']:.4f} P={res['precision']:.4f} R={res['recall']:.4f}")
    aucs=[s["auc"] for s in scores]
    print(f"AUC-ROC: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}")

if __name__=="__main__":
    mode=os.getenv("RUN_MODE","cv").lower()
    if mode=="train":
        train_and_predict_submission()
    else:
        cross_validation()


In [ ]:
!python triplet.py
%env RUN_MODE=train
!python deberta.py

In [ ]:
import pandas as pd
import numpy as np

# --------- 읽기 ---------
deb = pd.read_csv("deberta_submission.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "deb"})
tri = pd.read_csv("Triplet_submission.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "tri"})
q   = pd.read_csv("submission_qwen.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "q"})
q3  = pd.read_csv("submission_qwen3.csv")[["row_id", "rule_violation"]].rename(columns={"rule_violation": "q3"})

# --------- 교집합 머지 ---------
dfs = [deb, tri, q, q3]
df = dfs[0]
for d in dfs[1:]:
    df = df.merge(d, on="row_id", how="inner")

def minmax_scale(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan)
    fill = s.median()
    s = s.fillna(0.5 if pd.isna(fill) else fill)
    mn, mx = s.min(), s.max()
    if mx > mn:
        return (s - mn) / (mx - mn)
    return pd.Series(0.5, index=s.index, dtype=float)  # 상수 벡터면 중립값

# --------- 모델별 스케일링 ---------
for col in ["deb", "tri", "q", "q3"]:
    df[f"{col}_s"] = minmax_scale(df[col])

# --------- 가중치(필요시 수정) ---------
weights = {
    "deb_s": 0.43,
    "tri_s": 0.27,
    "q_s"  : 0.22,
    "q3_s" : 0.08,
}

# 안전장치: 합 1이 아니면 정규화
wsum = sum(weights.values())
if abs(wsum - 1.0) > 1e-8:
    weights = {k: v / wsum for k, v in weights.items()}

# --------- 앙상블 ---------
df["rule_violation"] = sum(weights[col] * df[col] for col in weights.keys())

# --------- 저장 ---------
out_cols = ["row_id", "rule_violation"]
df[out_cols].to_csv("submission.csv", index=False)

# 참고용 상관관계(스케일 후) 출력
print("Scaled predictions correlation:\n", df[[k for k in weights.keys()]].corr())
print(f"submission.csv saved with {len(df)} rows",
      f"(weights: {', '.join([f'{k}:{v:.2f}' for k,v in weights.items()])})")
